In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
!pip -q install tensorflow-datasets scikit-learn matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32
SEED = 42
CLASS_NAMES = ["rock", "paper", "scissors"]

AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
(ds_train_full, ds_test), ds_info = tfds.load(
    "rock_paper_scissors",
    split=["train", "test"],
    as_supervised=False,
    with_info=True
)

print(ds_info)

In [ ]:
def show_samples(dataset, n=9):
    plt.figure(figsize=(8,8))
    for i, example in enumerate(dataset.take(n)):
        img = example["image"].numpy()
        label = example["label"].numpy()
        plt.subplot(3,3,i+1)
        plt.imshow(img)
        plt.title(CLASS_NAMES[label])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(ds_train_full, n=9)

In [ ]:
def preprocess(example):
    img = tf.image.resize(example["image"], (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0   # normalize to [0,1]
    label = tf.cast(example["label"], tf.int32)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return img, label

In [ ]:
train_size = tf.data.experimental.cardinality(ds_train_full).numpy()
val_size = int(0.15 * train_size)

ds_train_full = ds_train_full.shuffle(5000, seed=SEED, reshuffle_each_iteration=True)
ds_val = ds_train_full.take(val_size)
ds_train = ds_train_full.skip(val_size)

ds_train = (ds_train
            .map(preprocess, num_parallel_calls=AUTOTUNE)
            .map(augment, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

ds_val = (ds_val
          .map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

ds_test = (ds_test
           .map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

print("Train batches:", tf.data.experimental.cardinality(ds_train).numpy())
print("Val batches:", tf.data.experimental.cardinality(ds_val).numpy())
print("Test batches:", tf.data.experimental.cardinality(ds_test).numpy())


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Stage 1: freeze backbone

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(3, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("rps_mobilenetv2.keras", save_best_only=True),
]

history1 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True

# Freeze most layers, unfreeze only the last ~30 for safe fine-tuning
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # smaller LR for fine-tuning
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=8,
    callbacks=callbacks
)

In [ ]:
def plot_history(histories, metric="accuracy"):
    plt.figure(figsize=(7,5))
    for i, h in enumerate(histories, start=1):
        plt.plot(h.history[metric], label=f"train_{metric}_stage{i}")
        plt.plot(h.history["val_" + metric], label=f"val_{metric}_stage{i}")
    plt.xlabel("Epoch")
    plt.ylabel(metric.capitalize())
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history([history1, history2], metric="accuracy")
plot_history([history1, history2], metric="loss")

In [ ]:
best_model = tf.keras.models.load_model("rps_mobilenetv2.keras")

test_loss, test_acc = best_model.evaluate(ds_test)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
y_true = []
y_pred = []

for imgs, labels in ds_test:
    preds = best_model.predict(imgs, verbose=0)
    preds_cls = np.argmax(preds, axis=1)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(preds_cls.tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix (Test Set)")
plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
for imgs, labels in ds_test.take(1):
    preds = best_model.predict(imgs, verbose=0)
    preds_cls = np.argmax(preds, axis=1)

    plt.figure(figsize=(10,10))
    for i in range(16):
        plt.subplot(4,4,i+1)
        plt.imshow((imgs[i].numpy() * 255).astype(np.uint8))
        t = CLASS_NAMES[int(labels[i].numpy())]
        p = CLASS_NAMES[int(preds_cls[i])]
        plt.title(f"True: {t}\nPred: {p}", fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()